# Lab 26: Model Context Protocol (MCP)

**Duration:** 3 hours 30 minutes. **Level:** intermediate. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Lab 25 you wrote the glue between the model and your functions yourself: the tool descriptions, the loop, the checks. Every new model or program needs the same glue again. The **Model Context Protocol (MCP)** is an open standard that removes this work. A program that **owns** tools or data runs as an **MCP server**. Any **MCP client** can connect to it, **discover** what it offers and **call** it, without special code. In this lab you turn the simulated building of Lab 25 into an MCP server, connect a client to it, let the language model use the tools through MCP, add a new tool without changing the client, and compare MCP with the direct way of Lab 25.

**By the end of this lab you can:**
1. Explain MCP: host, client, server, and the three things a server can offer (tools, resources, prompts).
2. Build a Python MCP server that offers tools, a resource and a prompt.
3. Connect an MCP client over stdio, discover what the server offers and call a tool.
4. Let the language model use MCP tools by turning the MCP tool list into tool descriptions for the model.
5. Compare MCP with direct tool calling, and name the safety rules for MCP servers.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Safety.** An MCP server is a program that runs on your computer and does things for the model. Start only servers that you wrote or that you trust, and give a server only the tools it needs. The building in this lab is simulated.

## 2. Background

**The problem.** Suppose 3 programs (a chat, an agent, an editor) must use 4 services (the building, a calendar, a database, a file store). With glue code for each pair that is 3 x 4 = 12 pieces of code, and every new program or service adds more. MCP puts a standard in the middle: each program speaks MCP **once**, each service speaks MCP **once**, and any program can use any service.

**The parts.**

| Part | What it is | In this lab |
|---|---|---|
| **Host** | The application with the language model and the user | Your notebook |
| **Client** | The part of the host that talks MCP to one server | `ClientSession` |
| **Server** | A program that offers tools, resources and prompts | `sensor_server.py` |
| **Transport** | How they talk | **stdio**: the client starts the server as a child process and they exchange JSON messages through its input and output |

**What a server offers.**

| Offer | Meaning | Example in this lab |
|---|---|---|
| **Tool** | A function the model can ask to run (it may change things) | `get_temperature`, `set_light` |
| **Resource** | Data the host can read, addressed by a name like a web address | `building://status` |
| **Prompt** | A ready-made prompt with arguments | `room_report(room)` |

**The conversation.** MCP messages are JSON (the protocol is called JSON-RPC). The client starts with `initialize`, then asks `tools/list`, `resources/list`, `prompts/list`, and calls `tools/call`. The server answers each. You do not write this JSON: the library does.

**MCP is not the model.** The model still uses **tool calling** (Lab 25). The host takes the list of tools from the MCP server, hands it to the model as tool descriptions, and when the model asks for a tool, the host forwards the request to the server. MCP standardises the part between the host and the tool provider.

| | Direct (Lab 25) | MCP (this lab) |
|---|---|---|
| Where the tools live | In the same program as the loop | In a separate server process |
| Adding a tool | Change the host code (the dictionary and the schema) | Change the server only. The client discovers it. |
| Other programs can use the tools | No | Yes, any MCP client |
| Cost | Simple, fast | An extra process and a protocol |

**Not always needed.** MCP is useful when tools are shared by many programs or written by other people. For one small program, direct tool calling is simpler. An agent (Lab 27) works with both.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(26)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: The Server of the Language Model and the Building
You need the language model server (Lab 20) and the simulated building (Lab 25). Run the cell.

In [ ]:
import asyncio
import inspect
import json
import subprocess
import sys
import time

import labview
import sensor_tools as building

print("Language model server ready:", labview.llm_ready())
print("Rooms of the building:", building.list_rooms())

### Step 2: Install the MCP Library
The cell installs the Python MCP library (version 2) if it is not there yet. It needs the internet, about 1 minute.

In [ ]:
try:
    import mcp
    print("mcp is installed")
except ImportError:
    print("Installing mcp ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp>=2,<3"], check=True)
    import mcp

from importlib.metadata import version
print("mcp version:", version("mcp"))

---
# Part B: Build an MCP Server (40 min)

### Step 1: Write the Server File
The cell writes the file `sensor_server.py`. Read it. It makes an `MCPServer` and uses decorators to offer functions as tools, one value as a resource and one prompt. The tool functions call the **same** functions of `sensor_tools` as in Lab 25. The line `mcp.run()` starts the server: it waits for messages on its standard input.

In [ ]:
%%writefile sensor_server.py
"""An MCP server for the simulated building (Lab 26)."""
import json

import sensor_tools as building
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("smart-building")


@mcp.tool()
def list_rooms() -> list[str]:
    """List the names of all rooms that have sensors."""
    return building.list_rooms()


@mcp.tool()
def get_temperature(room: str) -> float:
    """Read the temperature of a room in degrees Celsius."""
    return building.get_temperature(room)


@mcp.tool()
def get_humidity(room: str) -> float:
    """Read the relative humidity of a room in percent."""
    return building.get_humidity(room)


@mcp.tool()
def set_light(room: str, on: bool) -> str:
    """Switch the light of a room on (true) or off (false)."""
    return building.set_light(room, on)


@mcp.tool()
def set_fan(room: str, on: bool) -> str:
    """Switch the fan of a room on (true) or off (false). A fan makes the room cooler."""
    return building.set_fan(room, on)


@mcp.tool()
def open_door(name: str) -> str:
    """Unlock and open a door. The doors are: front, garage."""
    return building.open_door(name)


@mcp.resource("building://status")
def status() -> str:
    """The state of every room and door, as JSON."""
    return json.dumps({"rooms": building.STATE, "doors": building.DOORS})


@mcp.prompt()
def room_report(room: str) -> str:
    """A prompt that asks for a short report about one room."""
    return f"Write a two-sentence report about the {room}: its temperature, its humidity and whether the light and fan are on."


if __name__ == "__main__":
    mcp.run()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many tools, resources and prompts the server offers:
- Which line makes a function a tool:
- Where does the model read what a tool does: which part of the code becomes its description:

### Step 2: Check That the File Starts
The server talks through the standard input and output, so you cannot see it work in a cell. The cell only checks that the file has no mistakes: it compiles it and imports the module without starting it.

In [ ]:
import py_compile
py_compile.compile("sensor_server.py", doraise=True)
print("sensor_server.py compiles. Tools in the file:", [name for name in ("list_rooms", "get_temperature", "get_humidity", "set_light", "set_fan", "open_door") if f"def {name}" in open("sensor_server.py").read()])

---
# Part C: Connect a Client (40 min)

### Step 1: Start the Server from Python and Say Hello
`stdio_client` starts `sensor_server.py` as a child process. `ClientSession` speaks MCP with it. `initialize()` is the hello. The code is `async`: a notebook cell can use `await` directly. The helper `with_session` opens a session, runs a job and closes the server again.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["sensor_server.py"])


async def with_session(job):
    """Start the server, say hello, run job(session), and stop the server."""
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await job(session)


async def discover(session):
    tools = await session.list_tools()
    resources = await session.list_resources()
    prompts = await session.list_prompts()
    return tools.tools, resources.resources, prompts.prompts


tools, resources, prompts = await with_session(discover)
print("TOOLS")
for t in tools:
    print(f"  {t.name:<16} {list(t.input_schema['properties'])}   {t.description}")
print("RESOURCES:", [str(r.uri) for r in resources])
print("PROMPTS:  ", [p.name for p in prompts])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many tools did the client discover, and did you have to tell the client their names:
- Compare one tool description here with the description that building.schema made in Lab 25: what is the same:
- Where did the client get the list from:

### Step 2: Call a Tool, Read a Resource, Get a Prompt
The client calls `get_temperature`, reads the resource `building://status` and asks for the prompt `room_report`. A tool answers with a list of **content** pieces; the text is in the first one.

In [ ]:
async def try_things(session):
    result = await session.call_tool("get_temperature", {"room": "kitchen"})
    print("get_temperature ->", result.content[0].text, "| is_error:", result.is_error)
    result = await session.call_tool("set_light", {"room": "kitchen", "on": True})
    print("set_light       ->", result.content[0].text)
    status = await session.read_resource("building://status")
    print("status          ->", json.loads(status.contents[0].text)["rooms"]["kitchen"])
    prompt = await session.get_prompt("room_report", {"room": "lab"})
    print("prompt          ->", prompt.messages[0].content.text)


await with_session(try_things)
print("\nLight in the kitchen in THIS notebook (not in the server):", building.STATE["kitchen"]["light"])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did each of the four calls return:
- Is the light in the kitchen on in this notebook? Why not, when the tool set it on:
- Where does the state of the building live when you use MCP:

### Step 3: A Wrong Request
What does the server do with a room that does not exist, and with a wrong type? An MCP tool does not crash the server: it answers with `is_error`.

In [ ]:
async def try_wrong(session):
    for name, arguments in (("get_temperature", {"room": "moon base"}), ("get_temperature", {"room": 5}), ("get_temperature", {}), ("make_coffee", {})):
        try:
            result = await session.call_tool(name, arguments)
            print(f"{name}({arguments}) -> is_error={result.is_error}: {result.content[0].text.splitlines()[0][:100]}")
        except Exception as error:
            print(f"{name}({arguments}) -> exception {type(error).__name__}: {str(error)[:100]}")


await with_session(try_wrong)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which wrong requests were reported as an error result and which raised an exception:
- Who checked the arguments here: you (Lab 25) or the library? What did you save:

---
# Part D: Let the Language Model Use MCP Tools (45 min)

### Step 1: Turn MCP Tools into Tool Descriptions
The language model does not know MCP. The host must translate. An MCP tool has `name`, `description` and `input_schema`, which is exactly what the model needs. The function `to_model_tool` builds the description in the format of Lab 25.

In [ ]:
def to_model_tool(t):
    """An MCP tool description -> a tool description for the language model."""
    return {"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.input_schema}}


model_tools = [to_model_tool(t) for t in tools]
print(json.dumps(model_tools[1], indent=2))

### Step 2: The Loop with MCP
`ask_mcp` is the loop of Lab 25, with one change: the tools come from the **server** (`list_tools`), and a request is run by `session.call_tool`. Nothing in the loop knows what the tools do. Read the cell and compare it with `ask` in Lab 25.

In [ ]:
SYSTEM = "You help in a small smart building. Use the tools to read sensors and to switch things. Answer in one or two short sentences."


async def ask_mcp(question, max_rounds=5, show=True):
    """Ask a question; the model uses the tools of the MCP server. Returns the final answer."""
    async def job(session):
        listed = await session.list_tools()
        model_tools = [to_model_tool(t) for t in listed.tools]
        messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
        for round_number in range(1, max_rounds + 1):
            message = labview.llm_message(messages, tools=model_tools)["message"]
            messages.append(message)
            calls = message.get("tool_calls") or []
            if not calls:
                return (message.get("content") or "").strip()
            for call in calls:
                name, arguments = call["function"]["name"], json.loads(call["function"]["arguments"])
                result = await session.call_tool(name, arguments)
                text = ("ERROR: " if result.is_error else "") + (result.content[0].text if result.content else "")
                if show:
                    print(f"  round {round_number}: {name}({arguments}) -> {text[:80]}")
                messages.append({"role": "tool", "tool_call_id": call["id"], "content": text})
        return "(stopped: too many rounds)"
    return await with_session(job)


for question in ["What is the temperature in the kitchen?", "Switch on the lab light and tell me how humid the lab is.", "Which room is the warmest?", "Who wrote Hamlet?"]:
    print("Q:", question)
    print("A:", await ask_mcp(question), "\n")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which tools the model used for the question about the warmest room:
- What did the Hamlet question do:
- What part of the loop changed compared with Lab 25, and what stayed the same:

---
# Part E: Add a Tool Without Changing the Client (30 min)

### Step 1: A New Tool in the Server
In Lab 25, a new tool meant changing the host. With MCP you change the **server only**. The cell adds a function `get_dew_point` to the server file. The dew point is the temperature at which water starts to form on a surface. The simple formula `T - (100 - RH) / 5` is good enough for a classroom. Then it asks a question that needs the new tool. You do **not** change `ask_mcp`.

In [ ]:
source = open("sensor_server.py").read()
if "get_dew_point" not in source:
    new_tool = '''

@mcp.tool()
def get_dew_point(room: str) -> float:
    """Calculate the dew point of a room in degrees Celsius from its temperature and humidity."""
    return round(building.get_temperature(room) - (100 - building.get_humidity(room)) / 5, 1)


'''
    marker = "@mcp.resource"
    source = source.replace(marker, new_tool.lstrip("\n") + marker, 1)
    open("sensor_server.py", "w").write(source)
    print("get_dew_point was added to sensor_server.py")

tools, resources, prompts = await with_session(discover)
print("Tools now:", [t.name for t in tools])
print("A:", await ask_mcp("What is the dew point in the kitchen?"))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the client find the new tool without a change in ask_mcp:
- What is the dew point of the kitchen according to the model, and is it right (27.5 - (100 - 62) / 5 = 19.9):
- Which two files or cells would you have to change in Lab 25 to add the same tool:

### Step 2: Direct or MCP? A Measurement
The cell asks the same questions through MCP and through the direct loop of Lab 25, and measures the time. MCP starts a server process each time in this lab, so it is slower here. A real host keeps the server running.

In [ ]:
def run_tool(name, arguments):
    try:
        return json.dumps(building.FUNCTIONS[name](**arguments))
    except Exception as error:
        return f"ERROR: {error}"


def ask_direct(question, max_rounds=5):
    tools = building.all_schemas()
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_rounds):
        message = labview.llm_message(messages, tools=tools)["message"]
        messages.append(message)
        calls = message.get("tool_calls") or []
        if not calls:
            return (message.get("content") or "").strip()
        for call in calls:
            result = run_tool(call["function"]["name"], json.loads(call["function"]["arguments"]))
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return "(stopped)"


QUESTIONS = ["What is the temperature in the lab?", "How humid is the garage?"]
for question in QUESTIONS:
    started = time.time(); direct = ask_direct(question); t_direct = time.time() - started
    started = time.time(); through = await ask_mcp(question, show=False); t_mcp = time.time() - started
    print(f"{question}\n   direct {t_direct:.1f} s: {direct}\n   MCP    {t_mcp:.1f} s: {through}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Time direct and through MCP for the two questions:
- Are the answers the same:
- Where does the extra time come from:

---
# Part F: Safety, Test and Present (35 min)

### Step 1: What Should a Server Offer?
An MCP server gives a model the power to act. Decide what each tool is allowed to do. Fill in the table (double-click to edit).

| Tool | Reads or changes? | Risk if the model makes a mistake | Needs a person's approval? |
|---|---|---|---|
| get_temperature | | | |
| set_light | | | |
| set_fan | | | |
| open_door | | | |

**Rules for MCP servers.** (1) Run only servers you wrote or trust. (2) Give a server only the tools it needs (least privilege): a **read-only** server is much safer than one that can also switch things. (3) Validate every argument in the server, never trust the model. (4) Ask a person before a tool that changes the real world. (5) Do not put passwords in tool results.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which tools you would leave out of a server that a stranger can use:
- Which tool needs approval and why (you build the approval in Lab 27):

### Step 2: Test
Run the cell. It asks six questions through MCP and checks the building afterwards. Then fill in the table.

In [ ]:
TESTS = [("What is the temperature in the lab?", "get_temperature"), ("Switch off the lab light.", "set_light"), ("Which rooms exist?", "list_rooms"),
         ("How humid is the garage?", "get_humidity"), ("What is the dew point in the garage?", "get_dew_point"), ("Who wrote Hamlet?", None)]


async def tool_used(question):
    async def job(session):
        listed = await session.list_tools()
        reply = labview.llm_message([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], tools=[to_model_tool(t) for t in listed.tools])["message"]
        calls = reply.get("tool_calls") or []
        return calls[0]["function"]["name"] if calls else None
    return await with_session(job)


right = 0
for question, expected in TESTS:
    got = await tool_used(question)
    right += got == expected
    print(f"{'ok   ' if got == expected else 'WRONG'} {question:<45} expected {expected}  got {got}")
print(f"{right} of {len(TESTS)} right")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Score:
- Which question was wrong, and why do you think so:

### Step 3: Make It Your Own
Add **at least one improvement** from the list, or one of your own.

- Add a new tool to the server (for example `list_open_doors` or `get_comfort` that says too hot, too cold or fine) and ask a question that needs it.
- Add a second resource, `building://rooms`, that lists the rooms.
- Make the server **read-only**: create `sensor_server_readonly.py` with only the read tools, point `SERVER` to it and ask the model to switch a light. What happens?
- Print the JSON messages that the client sends: compare the `tools/call` request with what you called in Python.

> **Tip.** Change one thing at a time. After you change the server file, start a new session.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- What did you see:

### Step 4: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show the server file, the discovered tools, a call with the result, the language model using a tool through MCP, and a new tool that appears without a change in the client.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What the server offers (tools, resources, prompts):
- The score of the test:
- The time direct and through MCP:
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future (hint: Lab 27):

> **Where next.** You now have two ways to give tools to a model: directly (Lab 25) and through MCP (this lab). In Lab 27 an **agent** uses them to work through a task with several steps, with a step limit, memory of what it did, and a person's approval before it changes anything.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | `sensor_server.py` is written and compiles | |
| 2 | The client discovers the tools, the resource and the prompt, and you explain where the list comes from | |
| 3 | A tool is called, a resource is read and a prompt is fetched | |
| 4 | Wrong requests give an error result and you explain who checked them | |
| 5 | The language model answers with tools through MCP (`ask_mcp`) | |
| 6 | A new tool is added to the server and used without a change in the client | |
| 7 | The table of safety and approval is complete | |
| 8 | The test score is shown and at least one improvement works | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What problem does MCP solve? Use the 3 x 4 example.:
- 2. Name the host, the client and the server in this lab.:
- 3. What are the three things an MCP server can offer? Give an example of each.:
- 4. What is the transport in this lab, and how does the client start the server?:
- 5. How does the language model learn about an MCP tool? Which three fields of the tool does the host use?:
- 6. Why is the state of the building not in your notebook when you use MCP?:
- 7. In Lab 25, what must you change to add a tool? What do you change with MCP?:
- 8. Is MCP always better than direct tool calling? Give one case for each.:
- 9. Name two safety rules for MCP servers.:
- 10. Why should a tool that opens a door need a person's approval, even if the server checks the arguments?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `No module named 'mcp'` | Step 2 of Part A was not run, or there is no internet | Run Step 2 of Part A again. Check the internet with your teacher. |
| `No module named 'mcp.server.fastmcp'` | The old version 1 of the library is installed | Run `pip install 'mcp>=2,<3'` in a cell (with `!`), then restart the kernel |
| `ModuleNotFoundError: sensor_tools` when the server starts | The server runs with another Python | The notebook uses `sys.executable`. Check that you did not change `SERVER`. |
| `Connection closed` when you start a session | The server file has a mistake or does not exist | Run Step 1 and Step 2 of Part B again; start the server by hand in a Terminal to see the error: `python sensor_server.py` |
| `SyntaxError: ... await` outside function | The cell is not run in JupyterLab, or you pasted it in a script | Use `await` directly in a notebook cell. In a script use `asyncio.run(main())`. |
| The list of tools does not show my new tool | The old server process was used, or the file was not saved | Run the cell again. Every `with_session` starts a new server, so the file is read again. |
| A long text with `Traceback` and `UnexpectedToolError` appears | This is the log of the server: a tool raised an error (for example an unknown room). The client received an error result | This is normal. Read the last line. The model sees the error text and can try again |
| The answer is slow | Each question starts a new server process in this lab | This is normal here. A real host keeps the server running. |
| `Language model server ready: False` | The server of Lab 20 is not started | Ask the student who started it in Lab 20 to run `labview.llm_start()` |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(26)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Model Context Protocol: https://modelcontextprotocol.io/
- Python SDK for MCP: https://github.com/modelcontextprotocol/python-sdk
- The specification (messages, transports): https://modelcontextprotocol.io/specification